# CS156 Assignment #1

## 1. What am I trying to do? 

I have a personal assistant I use through messaging apps. Under the hood, it's an OpenClaw powered by OpenAI's GPT 5.6 Terra via the API. 

I have three issues with it: 
1. It costs me money, as OpenClaw does not draw from my OpenAI usage. 
2. I've found over time that I mostly use it for really simple things that I could have a local model do.
3. I'd ideally not have OpenAI process my personal information or any other cloud provider for that matter.

My main use case for the bot is setting reminders. I text it to remind me of everything I need to be on top of. 

To get a ballpark estimate of my usage for reminders, I used Claude Code on the virtual machine it lives on to count how many messages I sent during its lifetime that contained the word "remind" or "reminder". It counted **375** out of 693 messages. 

Since these reminder asks are extremely simple, it would be nice to have an intelligent classifier that decides whether they should go to OpenAI or to a local model without having to send the message to someone else's cloud. It'd fix all three issues stated above. 

Concretely, my problem is: build a classifier that decides whether a message is a reminder-related ask that I can use to route prompts to different models. 

The easiest way to tackle this is using a regular expression like `\bremind\w*` that looks for any variation of "remind" in a message (like "remind" or "reminder"), but that would be problematic if I ever include the word in my prompt out of context  or if I mention the word out of context. 

One level above that I could use a custom regex for a pattern like `!reminder` with `(?<!\S)!reminder\b` to avoid false positives, but that adds friction to me when typing.

Another level above that, many "System 1" general purpose classifier models came out recently, some of which can run locally. However, I don't think they could run in the tiny virtual machine my assistant lives on, so ideally I can use a smaller, simpler alternative.

I will compare my approach with each of these approaches at the end. 

## 2. Pipeline Diagram

## 3. The Data

The data consists of all the messages I've sent to my OpenClaw, filtered to messages that were sent 2 days after the first message, as those are mostly setup related. 

I extracted them from Telegram using their 'Chat export' feature on the Desktop app that makes a .html with all the conversation history between me and the agent. 

The notebook parses this history for turns where USER_NAME = "Stiven Triana" and writes them to the following schema in a csv:
```
    id                  SHA-256 hash of the message text
    message             the raw text I sent
    has_remind_keyword  1 if the text matches the regex \bremind\w*, else 0
    timestamp           when the message was sent (ISO 8601)
    label               reminder = 1, other = 0
```

I chose a hash-based id to have a unique way to distinguish messages from one another. has_remind_keyword is meant to serve as a baseline for counting and comparing the naive approach with other methods. label is NOT determined from has_remind_keyword, as some messages are reminder related but don't have 'remind' on them. I also removed individual messages that didn't have meaningful information with a filler word filter, i.e. if a message only consisted of 'yes' or 'go ahead'.

This left me with the following message counts:

| Stage | Messages |
|---|---|
| All messages I sent | 679 |
| After removing the first 2 days | 563 |
| After removing filler replies (e.g. "yes", "lgtm") | 559 |
| After de-duplicating by hash | 558 |
| Matching the `has_remind_keyword` regex | 238 |

The counts likely disagree with the earlier Claude Code estimate because it must have under/over counted due to failures in parsing.

I needed a way to ```label``` these messages that didn't send them to a cloud provider, didn't require me manually going through each, and would know when a message was a reminder even if it didn't contain remind on it (e.g. 'lmk at 7 my assignment is due').

I resorted to using a local LLM via Ollama for classification. I sent each message to the model with the same prompt, providing it some examples of messages that constitute a reminder, from the obvious to the ambiguous. The model's verdict resulted in the ```label``` column. 

The ```label``` column disagrees with the ```has_remind_keyword``` column #% of the time. Meaning the naive approach would've mis-classified # messages. Inspecting the disagreements up close... 



## 4. Repository

https://github.com/SomeoneElseSt/cs-156/tree/master/assignment-1

## 5. Novel Equations

## 6. Novel Algorithms/Metrics

## 7. Model Learning

## 8. Final Model Performance

## 9. Comparison

## 10. Discussion

## 11. References